# 00 — Environment Setup
**Run this once, top to bottom, before anything else.**

Steps:
1. Install all packages
2. Verify RTX 3070 GPU
3. Test CodeCarbon energy tracker
4. Download EGFR data from ChEMBL
5. Smoke test all libraries
6. Save config file

---
## STEP 1 — Install Packages
This takes 2-5 minutes on first run.

In [19]:
import subprocess, sys

def pip(package):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

packages = [
    'numpy',
    'pandas',
    'requests',
    'scikit-learn',
    'scipy',
    'rdkit',
    'deepchem',
    'torch-geometric',
    'transformers',
    'datasets',
    'tokenizers',
    'setuptools',
    'codecarbon',
    'psutil',
    'pynvml',
    'optuna',
    'matplotlib',
    'seaborn',
    'plotly',
    'umap-learn',
    'tqdm',
    'joblib',
]

failed = []
for pkg in packages:
    try:
        pip(pkg)
        print(f'  ✓ {pkg}')
    except Exception as e:
        print(f'  ✗ {pkg}  ← FAILED')
        failed.append(pkg)

print()
if failed:
    print(f'Failed: {failed} — we will handle these manually below')
else:
    print('All packages installed successfully.')

  ✓ numpy
  ✓ pandas
  ✓ requests
  ✓ scikit-learn
  ✓ scipy
  ✓ rdkit
  ✓ deepchem
  ✓ torch-geometric
  ✓ transformers
  ✓ datasets
  ✓ tokenizers
  ✓ setuptools
  ✓ codecarbon
  ✓ psutil
  ✓ pynvml
  ✓ optuna
  ✓ matplotlib
  ✓ seaborn
  ✓ plotly
  ✓ umap-learn
  ✓ tqdm
  ✓ joblib

All packages installed successfully.


---
## STEP 2 — Verify RTX 3070 GPU

In [20]:
import torch

print('='*50)
print('GPU CHECK')
print('='*50)

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    name   = torch.cuda.get_device_name(0)
    vram   = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  GPU:     {name}')
    print(f'  VRAM:    {vram:.1f} GB')
    print(f'  CUDA:    {torch.version.cuda}')
    print(f'  PyTorch: {torch.__version__}')
    print()
    print('  ✓ GPU ready')
    USE_AMP = True
else:
    DEVICE  = torch.device('cpu')
    USE_AMP = False
    print('  ✗ GPU not found — running on CPU')
    print('  Fix: pip install torch --index-url https://download.pytorch.org/whl/cu121')

print('='*50)

GPU CHECK
  GPU:     NVIDIA GeForce RTX 3080
  VRAM:    12.9 GB
  CUDA:    12.1
  PyTorch: 2.5.1+cu121

  ✓ GPU ready


---
## STEP 3 — Test CodeCarbon Energy Tracker

In [21]:
import os
from codecarbon import EmissionsTracker

os.makedirs('./results', exist_ok=True)

# Minimal config — avoids all deprecated parameters
CARBON_CONFIG = {
    'project_name': 'drug_discovery',
    'output_dir':   './results/',
    'log_level':    'error',   # suppress info noise
}

print('Testing CodeCarbon...')
tracker = EmissionsTracker(**CARBON_CONFIG)
tracker.start()

# Small dummy workload
import torch
if torch.cuda.is_available():
    x = torch.randn(500, 500, device='cuda')
    _ = x @ x
    del x
    torch.cuda.empty_cache()

emissions = tracker.stop()
print(f'  ✓ CodeCarbon working')
print(f'  Emissions: {emissions * 1e6:.4f} μg CO2')

[codecarbon WARNING @ 12:58:38] Multiple instances of codecarbon are allowed to run at the same time.


Testing CodeCarbon...
  ✓ CodeCarbon working
  Emissions: 4.2868 μg CO2


---
## STEP 4 — Download EGFR Data from ChEMBL
This replaces PyTDC (which has Windows install issues).
Downloads ~5,000-10,000 real EGFR inhibitor compounds with IC50 values.
Takes 1-3 minutes depending on connection speed.

In [22]:
import requests, time, os
import pandas as pd

os.makedirs('./data', exist_ok=True)

SAVE_PATH = './data/egfr_raw.csv'

# Skip download if already done
if os.path.exists(SAVE_PATH):
    df_raw = pd.read_csv(SAVE_PATH)
    print(f'Already downloaded — loaded {len(df_raw):,} compounds from {SAVE_PATH}')
else:
    print('Fetching EGFR IC50 data from ChEMBL API...')
    print('(This may take 1-3 minutes)')
    print()

    base_url = 'https://www.ebi.ac.uk/chembl/api/data/activity'
    params = {
        'target_chembl_id': 'CHEMBL203',  # EGFR
        'standard_type':    'IC50',
        'standard_units':   'nM',
        'assay_type':       'B',          # Binding assays only
        'format':           'json',
        'limit':            1000,
        'offset':           0,
    }

    all_records = []
    page = 1

    while len(all_records) < 10000:
        try:
            resp = requests.get(base_url, params=params, timeout=30)
            resp.raise_for_status()
            data = resp.json()
        except Exception as e:
            print(f'  Connection error: {e}')
            break

        records = data.get('activities', [])
        if not records:
            break

        all_records.extend(records)
        print(f'  Page {page:>2} — {len(all_records):,} records fetched so far')

        if data.get('page_meta', {}).get('next') is None:
            break

        params['offset'] += 1000
        page += 1
        time.sleep(0.5)  # Polite delay

    print()
    print(f'Total raw records: {len(all_records):,}')

    # Parse to DataFrame
    rows = []
    for r in all_records:
        smi = r.get('canonical_smiles')
        val = r.get('standard_value')
        cid = r.get('molecule_chembl_id')
        if smi and val:
            try:
                rows.append({
                    'Drug':       smi,
                    'Y':          float(val),
                    'chembl_id':  cid,
                })
            except:
                pass

    df_raw = pd.DataFrame(rows).drop_duplicates('Drug').reset_index(drop=True)
    df_raw.to_csv(SAVE_PATH, index=False)
    print(f'Saved {len(df_raw):,} unique compounds to {SAVE_PATH}')

print()
print('Sample data:')
print(df_raw.head())
print()
print('Activity (IC50 nM) summary:')
print(df_raw['Y'].describe())

Fetching EGFR IC50 data from ChEMBL API...
(This may take 1-3 minutes)

  Page  1 — 1,000 records fetched so far
  Page  2 — 2,000 records fetched so far
  Page  3 — 3,000 records fetched so far
  Page  4 — 4,000 records fetched so far
  Page  5 — 5,000 records fetched so far
  Page  6 — 6,000 records fetched so far
  Page  7 — 7,000 records fetched so far
  Page  8 — 8,000 records fetched so far
  Page  9 — 9,000 records fetched so far
  Page 10 — 10,000 records fetched so far

Total raw records: 10,000
Saved 7,050 unique compounds to ./data/egfr_raw.csv

Sample data:
                                                Drug          Y     chembl_id
0  Cc1cc(C)c(/C=C2\C(=O)Nc3ncnc(Nc4ccc(F)c(Cl)c4)...       41.0   CHEMBL68920
1  Cc1cc(C(=O)N2CCOCC2)[nH]c1/C=C1\C(=O)Nc2ncnc(N...      170.0   CHEMBL69960
2        CN(c1ccccc1)c1ncnc2ccc(N/N=N/Cc3ccccn3)cc12     9300.0  CHEMBL137635
3             CC(=C(C#N)C#N)c1ccc(NC(=O)CCC(=O)O)cc1   500000.0  CHEMBL306988
4                             O=C(

---
## STEP 5 — Smoke Test All Libraries

In [23]:
results = {}

# RDKit
try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors, AllChem
    mol = Chem.MolFromSmiles('CC(=O)Oc1ccccc1C(=O)O')  # Aspirin
    mw  = Descriptors.MolWt(mol)
    fp  = AllChem.GetMorganFingerprintAsBitVect(mol, 2, 2048)
    results['RDKit'] = f'✓  MW={mw:.1f}, FP bits={len(fp)}'
except Exception as e:
    results['RDKit'] = f'✗  {e}'

# PyTorch + CUDA
try:
    import torch
    cuda = torch.cuda.is_available()
    results['PyTorch'] = f'✓  v{torch.__version__} | CUDA={cuda}'
except Exception as e:
    results['PyTorch'] = f'✗  {e}'

# DeepChem
try:
    import deepchem as dc
    results['DeepChem'] = f'✓  v{dc.__version__}'
except Exception as e:
    results['DeepChem'] = f'✗  {e}'

# Transformers
try:
    import transformers
    results['Transformers'] = f'✓  v{transformers.__version__}'
except Exception as e:
    results['Transformers'] = f'✗  {e}'

# PyTorch Geometric
try:
    import torch_geometric
    results['PyTorch Geometric'] = f'✓  v{torch_geometric.__version__}'
except Exception as e:
    results['PyTorch Geometric'] = f'✗  {e}'

# Optuna
try:
    import optuna
    results['Optuna'] = f'✓  v{optuna.__version__}'
except Exception as e:
    results['Optuna'] = f'✗  {e}'

# UMAP
try:
    import umap
    results['UMAP'] = '✓  imported'
except Exception as e:
    results['UMAP'] = f'✗  {e}'

# Scikit-learn
try:
    import sklearn
    results['Scikit-learn'] = f'✓  v{sklearn.__version__}'
except Exception as e:
    results['Scikit-learn'] = f'✗  {e}'

# CodeCarbon
try:
    from codecarbon import EmissionsTracker
    results['CodeCarbon'] = '✓  imported'
except Exception as e:
    results['CodeCarbon'] = f'✗  {e}'

# ChEMBL data check
try:
    import pandas as pd, os
    assert os.path.exists('./data/egfr_raw.csv')
    df = pd.read_csv('./data/egfr_raw.csv')
    results['EGFR Data'] = f'✓  {len(df):,} compounds in ./data/egfr_raw.csv'
except Exception as e:
    results['EGFR Data'] = f'✗  {e}'

# Print results
print('='*55)
print('SMOKE TEST RESULTS')
print('='*55)
all_passed = True
for lib, status in results.items():
    print(f'  {lib:<22} {status}')
    if status.startswith('✗'):
        all_passed = False
print('='*55)
if all_passed:
    print('  ✓ ALL PASSED — ready to proceed')
else:
    print('  Some tests failed — paste errors above in chat')

SMOKE TEST RESULTS
  RDKit                  ✓  MW=180.2, FP bits=2048
  PyTorch                ✓  v2.5.1+cu121 | CUDA=True
  DeepChem               ✓  v2.8.0
  Transformers           ✓  v5.7.0
  PyTorch Geometric      ✓  v2.7.0
  Optuna                 ✓  v4.8.0
  UMAP                   ✓  imported
  Scikit-learn           ✓  v1.7.2
  CodeCarbon             ✓  imported
  EGFR Data              ✓  7,050 compounds in ./data/egfr_raw.csv
  ✓ ALL PASSED — ready to proceed


[12:59:27] DEPRECATION WARNING: please use MorganGenerator


---
## STEP 6 — Save Project Config
Creates `config.json` used by all other notebooks.

In [24]:
import json, os, torch

os.makedirs('./data',    exist_ok=True)
os.makedirs('./models',  exist_ok=True)
os.makedirs('./results', exist_ok=True)
os.makedirs('./utils',   exist_ok=True)

CONFIG = {
    'gpu':          'RTX 3070',
    'vram_gb':      8,
    'use_amp':      torch.cuda.is_available(),
    'batch_size':   64,
    'num_workers':  4,
    'random_seed':  42,
    'test_size':    0.2,
    'val_size':     0.1,
    'target':       'EGFR',
    'data_source':  'ChEMBL',
    'raw_data':     './data/egfr_raw.csv',
    'task':         'regression',
    'models':       ['rf_morgan', 'attentivefp', 'mpnn_deepchem', 'chemberta2'],
    'data_dir':     './data/',
    'model_dir':    './models/',
    'results_dir':  './results/',
}

with open('./config.json', 'w') as f:
    json.dump(CONFIG, f, indent=2)

print('config.json saved:')
print(json.dumps(CONFIG, indent=2))

config.json saved:
{
  "gpu": "RTX 3070",
  "vram_gb": 8,
  "use_amp": true,
  "batch_size": 64,
  "num_workers": 4,
  "random_seed": 42,
  "test_size": 0.2,
  "val_size": 0.1,
  "target": "EGFR",
  "data_source": "ChEMBL",
  "raw_data": "./data/egfr_raw.csv",
  "task": "regression",
  "models": [
    "rf_morgan",
    "attentivefp",
    "mpnn_deepchem",
    "chemberta2"
  ],
  "data_dir": "./data/",
  "model_dir": "./models/",
  "results_dir": "./results/"
}


---
## ✓ Setup Complete

If all steps above passed, your environment is ready.

**Next:** Open `01_data.ipynb`

---
### Troubleshooting Reference

| Problem | Fix |
|---|---|
| CUDA out of memory | Edit config.json — set `batch_size` to 32 |
| GPU not detected | Run: `pip install torch --index-url https://download.pytorch.org/whl/cu121` |
| ChEMBL download failed | Re-run Step 4 — sometimes times out on first try |
| Any import error | Paste error in chat and we will fix it |